# Task 1 — Build the Pipeline

In [16]:
import requests
import time
import pandas as pd
import sqlite3
import json

############## reviewer to replace it with their own key ############

from google.colab import userdata
tmdb_api_key = userdata.get('tmdb_api_key')


params = {
    "api_key": tmdb_api_key,
    "language": "en-US",
    "page": 1
}

Hit API and Store response in form of a JSON

In [17]:
response = requests.get("https://api.themoviedb.org/3/discover/movie", params=params)
data = response.json()

Store JSON results into a Dataframe

In [18]:
movies_df = pd.DataFrame(data["results"])
movies_df["genre_ids"] = movies_df["genre_ids"].apply(json.dumps)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 100)

Insert dataframe data into a SQL database named movies.

In [19]:
conn = sqlite3.connect("tmdb.db")
movies_df.to_sql("movies", conn, if_exists="replace", index=False)
print("Data inserted into SQLite database successfully.")

Data inserted into SQLite database successfully.


# Task 2 — Perform EDA

Load SQL Database data back into Dataframe for EDA

In [20]:
conn = sqlite3.connect("tmdb.db")
df = pd.read_sql("SELECT * FROM movies", conn)
df["adult"] = df["adult"].astype(bool)
df["video"] = df["video"].astype(bool)


a. Display the first 5 rows

In [21]:
print(df.head())

   adult                     backdrop_path        genre_ids       id original_language              original_title                                                                                             overview  popularity                       poster_path release_date                       title  video  vote_average  vote_count
0  False  /1x9e0qWonw634NhIsRdvnneeqvN.jpg      [10749, 18]  1523145                ru   Твоё сердце будет разбито  High school student Polina is saved from bullying at her new school and makes a deal with the ma...   1318.7141  /7wIBfBl2gejt6xHxNSK0reVIm7E.jpg   2026-03-26   Your Heart Will Be Broken  False         6.891          32
1  False  /iN41Ccw4DctL8npfmYg1j5Tr1eb.jpg    [878, 12, 14]    83533                en        Avatar: Fire and Ash  In the wake of the devastating war against the RDA and the loss of their eldest son, Jake Sully ...    499.3595  /bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg   2025-12-17        Avatar: Fire and Ash  False         7.341     

b. Show summary statistics for numeric columns

In [22]:
print(df.describe())

                 id   popularity  vote_average   vote_count
count  2.000000e+01    20.000000     20.000000     20.00000
mean   1.017887e+06   285.960985      7.151950   1007.95000
std    3.450627e+05   262.461757      0.689053   2272.30392
min    8.353300e+04   120.952600      5.918000      2.00000
25%    8.462030e+05   142.207025      6.764000    123.25000
50%    1.137552e+06   236.629050      7.027000    414.00000
75%    1.271912e+06   296.408400      7.587250    785.75000
max    1.523145e+06  1318.714100      8.500000  10402.00000


c. Count the number of movies per genre

In [23]:
df["genre_ids"] = df["genre_ids"].apply(json.loads)
df_exploded = df.explode("genre_ids")
genre_counts = df_exploded["genre_ids"].value_counts()
print(genre_counts)

genre_ids
35       7
12       7
878      6
53       6
80       5
18       5
10751    5
28       4
16       4
10749    3
14       3
27       2
10402    1
9648     1
Name: count, dtype: int64


d. Identify columns with missing values

In [24]:
print(df.isnull().sum())

adult                0
backdrop_path        0
genre_ids            0
id                   0
original_language    0
original_title       0
overview             0
popularity           0
poster_path          0
release_date         0
title                0
video                0
vote_average         0
vote_count           0
dtype: int64
